# Origin-specific sex gap in stochastic accessibility

This notebook creates a publication-ready three-panel UTAM map for employment, education, and health. It uses the **frozen Gate 11 point estimates** and does not refit kernels, redefine targets, or rerun inference.

The mapped quantity is $h_F(i)-h_M(i)$: the difference between female- and male-specific expected first-passage transitions from the same origin UTAM $i$, under the primary specification (`core_50`, outside-target conditioning, $\tau=20$). Values are dimensionless aggregate mobility transitions, not minutes or individual itineraries.


In [ ]:
%pip -q install geopandas pyarrow matplotlib

from pathlib import Path
from datetime import datetime, timezone
import hashlib, json, os, re, shutil, unicodedata, zipfile

import geopandas as gpd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
import numpy as np
import pandas as pd

ROOT = Path('/content/colombia-stochastic-access')
DRIVE_BASE = Path('/content/drive/MyDrive/colombia-stochastic-access-data/bogota_em2023')
PURPOSES = ('employment', 'education', 'health')
PURPOSE_LABELS = {'employment': 'Employment', 'education': 'Education', 'health': 'Health'}
FIGURE_STEM = 'figure_origin_specific_sex_gap_map'

DRIVE_AVAILABLE = False
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    DRIVE_AVAILABLE = Path('/content/drive/MyDrive').exists()
except Exception as exc:
    print(f'Drive mount skipped: {exc}')

RESULTS_CSV = DRIVE_BASE / 'results/gate11/female_male_origin_contributions.csv'
ZONING_CACHE = DRIVE_BASE / 'zoning_edoh.zip'

if not (RESULTS_CSV.exists() and ZONING_CACHE.exists()):
    try:
        from google.colab import files
    except ImportError as exc:
        raise FileNotFoundError('Drive inputs are unavailable and manual upload requires Google Colab.') from exc
    print('Drive inputs were not found. Select both files in the same upload dialog:')
    print('  1) female_male_origin_contributions.csv (Gate 11)')
    print('  2) zoning_edoh.zip')
    uploaded = files.upload()
    staging = Path('/content/origin_gap_inputs')
    staging.mkdir(parents=True, exist_ok=True)
    csv_candidates, zip_candidates = [], []
    for name, content in uploaded.items():
        destination = staging / Path(name).name
        destination.write_bytes(content)
        del content
        if destination.suffix.lower() == '.csv':
            try:
                columns = set(pd.read_csv(destination, nrows=3).columns)
                if {'purpose_group', 'utam', 'female_fpt_steps', 'male_fpt_steps'} <= columns:
                    csv_candidates.append(destination)
            except Exception:
                pass
        elif destination.suffix.lower() == '.zip' and zipfile.is_zipfile(destination):
            zip_candidates.append(destination)
    if len(csv_candidates) != 1 or len(zip_candidates) != 1:
        raise ValueError('Upload exactly one valid Gate 11 CSV and one valid zoning ZIP.')
    RESULTS_CSV, ZONING_CACHE = csv_candidates[0], zip_candidates[0]

OUT_PRIMARY = (DRIVE_BASE / 'publication/origin_specific_sex_gap_map' if DRIVE_AVAILABLE else Path('/content/origin_specific_sex_gap_map'))
OUTPUT_FOLDERS = [OUT_PRIMARY]
if ROOT.exists():
    OUTPUT_FOLDERS.append(ROOT / 'outputs/figures')
for folder in OUTPUT_FOLDERS:
    folder.mkdir(parents=True, exist_ok=True)

print('Frozen estimates:', RESULTS_CSV)
print('Zoning archive:', ZONING_CACHE)
print('Output directory:', OUT_PRIMARY)


In [ ]:
def normalize(value):
    value = unicodedata.normalize('NFKD', str(value))
    value = ''.join(char for char in value if not unicodedata.combining(char))
    return re.sub(r'[^a-z0-9]+', '_', value.lower().strip()).strip('_')

def canonical_utam(series):
    def canonical(value):
        if pd.isna(value):
            return pd.NA
        text = str(value).strip().upper().replace(' ', '')
        upr = re.fullmatch(r'UPR0*(\d+)', text)
        if upr:
            return f'UPR{int(upr.group(1)):04d}'
        utam = re.fullmatch(r'(?:UTAM)?0*(\d+)', text)
        if utam:
            return f'UTAM{int(utam.group(1)):03d}'
        return text
    return series.astype('string').map(canonical).astype('string')

def sha256(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b''):
            digest.update(chunk)
    return digest.hexdigest()

extract_dir = Path('/content/zoning_edoh')
if extract_dir.exists():
    shutil.rmtree(extract_dir)
extract_dir.mkdir(parents=True)
with zipfile.ZipFile(ZONING_CACHE) as archive:
    root = extract_dir.resolve()
    for member in archive.infolist():
        resolved = (extract_dir / member.filename).resolve()
        if root != resolved and root not in resolved.parents:
            raise ValueError(f'Unsafe archive path: {member.filename}')
    archive.extractall(extract_dir)

spatial_paths = sorted(list(extract_dir.rglob('*.shp')) + list(extract_dir.rglob('*.geojson')) + list(extract_dir.rglob('*.gpkg')))
candidates = []
for path in spatial_paths:
    try:
        preview = gpd.read_file(path, rows=5)
        columns = {normalize(column): column for column in preview.columns}
        if 'utam' in columns:
            candidates.append((path, columns['utam']))
    except Exception:
        pass
if not candidates:
    raise FileNotFoundError('No spatial layer containing a UTAM field was found.')

utam_path, utam_column = sorted(candidates, key=lambda item: ('utam2023' not in normalize(item[0].stem), len(item[0].parts)))[0]
zones_raw = gpd.read_file(utam_path)
zones_raw['utam'] = canonical_utam(zones_raw[utam_column])
zones = zones_raw.dropna(subset=['utam']).dissolve(by='utam', as_index=False)

results = pd.read_csv(RESULTS_CSV, dtype={'utam': 'string'})
expected = {'purpose_group', 'utam', 'female_fpt_steps', 'male_fpt_steps'}
absent = sorted(expected - set(results.columns))
if absent:
    raise KeyError(f'Missing Gate 11 columns: {absent}')
results['utam'] = canonical_utam(results['utam'])
results['sex_gap_fpt_steps'] = results['female_fpt_steps'] - results['male_fpt_steps']
results['is_target_or_zero_gap'] = np.isclose(results['female_fpt_steps'], 0) & np.isclose(results['male_fpt_steps'], 0)

mapped = zones.merge(results, on='utam', how='inner', validate='one_to_many')
coverage = results['utam'].isin(zones['utam']).mean()
if coverage < 0.95:
    raise RuntimeError(f'Spatial coverage below 95%: {coverage:.1%}')

municipality_column = next((column for column in zones_raw.columns if normalize(column) in {'municipio', 'municipality', 'nom_mpio', 'nom_municipio'}), None)
municipal_boundaries = None
if municipality_column is not None:
    municipal_boundaries = zones_raw.dissolve(by=municipality_column, as_index=False).boundary

print('UTAM layer:', utam_path)
print('Mapped origin coverage:', f'{coverage:.1%}')
print('Municipality field:', municipality_column or 'not available')


In [ ]:
mpl.rcParams.update({
    'font.family': 'DejaVu Sans', 'font.size': 10, 'axes.titlesize': 12,
    'figure.titlesize': 17, 'savefig.facecolor': 'white',
})

non_target = mapped.loc[~mapped['is_target_or_zero_gap'], 'sex_gap_fpt_steps'].dropna().abs()
limit = float(non_target.quantile(0.98))
if not np.isfinite(limit) or limit <= 0:
    limit = float(non_target.max()) if len(non_target) else 1.0
norm = TwoSlopeNorm(vmin=-limit, vcenter=0.0, vmax=limit)
cmap = mpl.colormaps['RdBu_r'].copy()
cmap.set_bad('#E5E7EB')

fig, axes = plt.subplots(1, 3, figsize=(15.8, 7.0), constrained_layout=False)
for axis, purpose in zip(axes, PURPOSES):
    layer = mapped[mapped['purpose_group'].eq(purpose)].copy()
    layer.loc[layer['is_target_or_zero_gap'], 'sex_gap_fpt_steps'] = np.nan
    layer.plot(column='sex_gap_fpt_steps', cmap=cmap, norm=norm, linewidth=0.18, edgecolor='white', ax=axis, missing_kwds={'color': '#E5E7EB', 'edgecolor': 'white', 'hatch': '///'})
    if municipal_boundaries is not None:
        municipal_boundaries.plot(ax=axis, color='#263238', linewidth=0.65, alpha=0.9)
    axis.set_title(PURPOSE_LABELS[purpose], fontweight='semibold', pad=8)
    axis.set_axis_off()

fig.suptitle('Origin-specific sex gap in stochastic accessibility', x=0.5, y=0.965, fontweight='bold')
fig.text(0.5, 0.915, r'Female minus male expected first-passage transitions by origin UTAM ($\tau=20$)', ha='center', color='#455A64', fontsize=11)
fig.subplots_adjust(left=0.02, right=0.98, top=0.875, bottom=0.17, wspace=0.025)

sm = mpl.cm.ScalarMappable(norm=norm, cmap=cmap)
sm.set_array([])
cbar_ax = fig.add_axes([0.25, 0.105, 0.50, 0.026])
cbar = fig.colorbar(sm, cax=cbar_ax, orientation='horizontal', extend='both')
cbar.set_label('Female − male expected FPT (transitions)', labelpad=5)
fig.text(0.5, 0.035, 'Positive values indicate longer expected stochastic access for women. Hatched UTAMs are purpose targets or have zero FPT for both groups. FPT transitions are dimensionless and describe fitted aggregate mobility systems.', ha='center', va='bottom', fontsize=8.5, color='#546E7A', wrap=True)

for folder in OUTPUT_FOLDERS:
    fig.savefig(folder / f'{FIGURE_STEM}.png', dpi=400, bbox_inches='tight')
    fig.savefig(folder / f'{FIGURE_STEM}.pdf', bbox_inches='tight')
    fig.savefig(folder / f'{FIGURE_STEM}.eps', bbox_inches='tight')
plt.show()

mapped_export = mapped.drop(columns='geometry').sort_values(['purpose_group', 'utam'])
mapped_export.to_csv(OUT_PRIMARY / f'{FIGURE_STEM}_data.csv', index=False)
metadata = {
    'generated_utc': datetime.now(timezone.utc).isoformat(),
    'estimand': 'origin-specific female minus male expected first-passage transitions',
    'primary_specification': {'target_definition': 'core_50', 'conditioning': 'outside_target', 'tau': 20},
    'input': str(RESULTS_CSV), 'input_sha256': sha256(RESULTS_CSV),
    'zoning_archive_sha256': sha256(ZONING_CACHE), 'spatial_coverage': float(coverage),
    'shared_symmetric_limit': limit, 'outputs': ['png', 'pdf', 'eps', 'csv'],
}
(OUT_PRIMARY / f'{FIGURE_STEM}_metadata.json').write_text(json.dumps(metadata, indent=2, ensure_ascii=False, allow_nan=False), encoding='utf-8')

archive_path = shutil.make_archive(str(OUT_PRIMARY / FIGURE_STEM), 'zip', OUT_PRIMARY)
print('Publication files:', OUT_PRIMARY)
print('ZIP:', archive_path)
if ROOT.exists():
    print('Repository copies:', ROOT / 'outputs/figures')
if not DRIVE_AVAILABLE:
    from google.colab import files
    files.download(archive_path)


## Customisation

The main editable choices are concentrated in the final cell: `figsize`, `RdBu_r`, the 0.98 clipping quantile, title text, municipality outline width, and export DPI. Keep the same symmetric colour scale across panels so that magnitudes remain comparable.
